In [1]:
import pandas
import torch
from hugsvision.dataio.VisionDataset import VisionDataset
from hugsvision.nnet.VisionClassifierTrainer import VisionClassifierTrainer
from transformers import ViTFeatureExtractor, ViTForImageClassification, AutoImageProcessor
from hugsvision.inference.VisionClassifierInference import VisionClassifierInference

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

In [2]:
from transformers import TrainingArguments
import inspect

_orig_ta_init = TrainingArguments.__init__

def _patched_ta_init(self, *args, **kwargs):
    # 1) Compat: evaluation_strategy -> eval_strategy (tu parche anterior)
    if "evaluation_strategy" in kwargs and "eval_strategy" not in kwargs:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    # 2) Si metric_for_best_model pide eval_accuracy pero no tendremos accuracy,
    #    cámbialo a eval_loss y marca menor-es-mejor
    if kwargs.get("metric_for_best_model") == "eval_accuracy":
        kwargs["metric_for_best_model"] = "eval_loss"
        kwargs["greater_is_better"] = False
    return _orig_ta_init(self, *args, **kwargs)

TrainingArguments.__init__ = _patched_ta_init


In [3]:
#data preaparatoion 
train, val, id2label, label2id = VisionDataset.fromImageFolder(
    "./train/",
    test_ratio=0.1,
    balanced=True,
    augmentation=True,
    torch_vision=False
)

Split Datasets...
Balance train dataset...
The less represented label in train as 100 occurrences
Size of train after balancing is 300
Training Dataset Elements:  270
+---------+---------------+-------+-------+-------+
| Dataset | affenpinscher | akita | corgi | Total |
+---------+---------------+-------+-------+-------+
|  Train  |      91       |  87   |  92   |  270  |
|  Test   |       9       |  13   |   8   |  30   |
+---------+---------------+-------+-------+-------+


In [4]:
huggingface_model = 'google/vit-base-patch16-224-in21k'


In [5]:
trainer = VisionClassifierTrainer(
	model_name   = "MyDogClassifier",
	train        = train,
	test         = val,
	output_dir   = "./out/",
	max_epochs   = 20,
	batch_size   = 4, 
	lr	     = 2e-5,
	fp16	     = True,
    eval_metric="eval_loss",
	model = ViTForImageClassification.from_pretrained(
	    huggingface_model,
	    num_labels = len(label2id),
	    label2id   = label2id,
	    id2label   = id2label
	),
	feature_extractor = ViTFeatureExtractor.from_pretrained(
		huggingface_model,
	),
)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\wail-\.conda\envs\ml\lib\site-packages\transformers\models\vit\feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


{'0': 'affenpinscher', '1': 'akita', '2': 'corgi'}
{'affenpinscher': '0', 'akita': '1', 'corgi': '2'}
Trainer builded!
Start Training!


Epoch,Training Loss,Validation Loss
1,No log,0.356978
2,No log,0.105798
3,No log,0.060975
4,No log,0.046257
5,No log,0.038110
6,No log,0.032887
7,No log,0.029198
8,0.158200,0.026409
9,0.158200,0.024260
10,0.158200,0.022519


Model saved at: ./out/MYDOGCLASSIFIER/20_2025-09-28-01-57-42


In [6]:
y_true, y_pred = trainer.evaluate_f1_score()


100%|██████████| 30/30 [00:00<00:00, 33.00it/s]

               precision    recall  f1-score   support

affenpinscher     1.0000    1.0000    1.0000         9
        akita     1.0000    1.0000    1.0000        13
        corgi     1.0000    1.0000    1.0000         8

     accuracy                         1.0000        30
    macro avg     1.0000    1.0000    1.0000        30
 weighted avg     1.0000    1.0000    1.0000        30

Logs saved at: ./out/MYDOGCLASSIFIER/20_2025-09-28-01-57-42


In [7]:
cm = confusion_matrix(y_true, y_pred)
labels = list(label2id.keys())
df_cm = pd.DataFrame(cm, index = labels, columns = labels)

In [8]:
sns.heatmap(df_cm, annot=True, annot_kws={"size": 8}, fmt="")
plt.savefig("./conf_matrix_1.jpg")

In [ ]:
import os.path
base = "./out/MyDogClassifier/20_2025-09-28-01-57-42"
path_feat  = f"{base}/feature_extractor"  
path_model = f"{base}/model"              


img = "./test/affenpinscher/affenpinscher_0.jpg"


classifier = VisionClassifierInference(
    feature_extractor = AutoImageProcessor.from_pretrained(path_feat),
    model = ViTForImageClassification.from_pretrained(path_model),
)


label = classifier.predict(img_path=img)
print("Predicted class:", label)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
c:\Users\wail-\.conda\envs\ml\lib\site-packages\transformers\models\vit\feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


Model loaded!
Predicted class: affenpinscher


In [14]:
test_files = [f for f in glob.glob("./test/**/*", recursive=True) if os.path.isfile(f)]

print(f"Found {len(test_files)} test images.")

for fpath in test_files:
    
    true_label = os.path.basename(os.path.dirname(fpath))

   
    pred = classifier.predict(img_path=fpath)

   
    print(f"{fpath} | true: {true_label:15s} | pred: {pred}")


Found 60 test images.
./test\affenpinscher\affenpinscher_0.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_1.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_10.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_11.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_12.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_13.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_14.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_15.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_16.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_17.jpg | true: affenpinscher   | pred: affenpinscher
./test\affenpinscher\affenpinscher_18.jpg | true: affenpinscher   | pred: affenpinscher
./test\affen